# Inspect the BAF60 computeMatrix result

Each row is one MACS2 summit that passed `q <= 0.05`. The first six columns describe the region; the remaining columns are the `log2(IP/Input)` signal in 20-bp bins around the summit center.

In [ ]:
from pathlib import Path
import gzip, json
import pandas as pd

matrix_path = Path('/data1/home/zhangzhhui03/workspace/projects/SWP73B/ChIP_ATAC_MNase_PRJNA351855/results/chip_workflow/compute_matrix/BAF60.matrix.gz')
with gzip.open(matrix_path, 'rt') as f:
    metadata = json.loads(f.readline().removeprefix('@'))

bin_size = metadata['bin size'][0]
bin_starts = range(-metadata['upstream'][0], metadata['downstream'][0], bin_size)
columns = ['chrom', 'start', 'end', 'name', 'score', 'strand'] + [f'signal_{x}_{x + bin_size}' for x in bin_starts]
df = pd.read_csv(matrix_path, sep='\t', comment='@', header=None, names=columns)

print(df.columns.tolist())
df.head()

## Summit-centered heatmap and meta profile

Keep all called summits. Rows are only sorted by their mean signal within 200 bp of the summit; no additional low-signal filtering is applied.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.arange(-metadata['upstream'][0], metadata['downstream'][0], bin_size) + bin_size / 2
signal = df.iloc[:, 6:].to_numpy(float)
center = np.abs(x) <= 200
order = np.argsort(np.nanmean(signal[:, center], axis=1))[::-1]
color_limit = np.nanpercentile(np.abs(signal), 98)

fig, (ax_heatmap, ax_meta) = plt.subplots(2, 1, figsize=(7, 8), sharex=True, gridspec_kw={'height_ratios': [4, 1]})
image = ax_heatmap.imshow(signal[order], aspect='auto', cmap='RdBu_r', vmin=-color_limit, vmax=color_limit, extent=[-metadata['upstream'][0], metadata['downstream'][0], len(signal), 0])
ax_heatmap.axvline(0, color='black', lw=0.8)
ax_heatmap.set(ylabel='MACS2 summits', title=f'BAF60 summit-centered heatmap (n={len(signal):,})')
fig.colorbar(image, ax=ax_heatmap, label='log2(IP/Input)')

ax_meta.plot(x, np.nanmean(signal, axis=0), color='black')
ax_meta.axvline(0, color='grey', lw=0.8)
ax_meta.axhline(0, color='grey', lw=0.5)
ax_meta.set(xlabel='Distance from summit (bp)', ylabel='Mean log2(IP/Input)')
plt.tight_layout()

## Genomic annotation of MACS2 peaks

Use the full `narrowPeak` intervals and standard `gene`/`exon` features in the GTF. Categories are mutually exclusive with priority **Exon > Intron > strand-aware upstream 500 bp > Intergenic**. Genome fraction is the fraction of TAIR10 bases assigned by exactly the same rules.

In [ ]:
gtf_path = Path('/data1/home/zhangzhhui03/workspace/reference/Araport11/Araport11_GTF_genes_transposons.20241001.gtf')
peak_path = Path('/data1/home/zhangzhhui03/workspace/projects/SWP73B/ChIP_ATAC_MNase_PRJNA351855/results/chip_workflow/macs2/BAF60/BAF60_peaks.narrowPeak')
chrom_sizes_path = Path('/data1/home/zhangzhhui03/workspace/repos/ChIP_workflow/config/TAIR10.chrom.sizes')

chrom_sizes = dict(pd.read_csv(chrom_sizes_path, sep='\t', header=None).itertuples(index=False, name=None))
gtf = pd.read_csv(gtf_path, sep='\t', comment='#', header=None, usecols=[0, 2, 3, 4, 6],
                  names=['chrom', 'feature', 'start', 'end', 'strand'])
gtf['start'] -= 1  # GTF: 1-based inclusive -> BED/Python: 0-based half-open
genes = gtf[gtf['feature'] == 'gene']
exons = gtf[gtf['feature'] == 'exon']

# 0=Exon, 1=Intron, 2=Upstream 500, 3=Intergenic; later assignments have higher priority.
genome_label = {chrom: np.full(size, 3, dtype=np.uint8) for chrom, size in chrom_sizes.items()}
for chrom, start, end, strand in genes[['chrom', 'start', 'end', 'strand']].itertuples(index=False, name=None):
    if strand == '+':
        genome_label[chrom][max(0, start - 500):start] = 2
    else:
        genome_label[chrom][end:min(chrom_sizes[chrom], end + 500)] = 2
for chrom, start, end in genes[['chrom', 'start', 'end']].itertuples(index=False, name=None):
    genome_label[chrom][start:end] = 1
for chrom, start, end in exons[['chrom', 'start', 'end']].itertuples(index=False, name=None):
    genome_label[chrom][start:end] = 0

In [ ]:
categories = ['Exon', 'Intron', 'Upstream 500', 'Intergenic']
peaks = pd.read_csv(peak_path, sep='\t', header=None, usecols=[0, 1, 2], names=['chrom', 'start', 'end'])
peaks['category'] = [categories[int(genome_label[c][s:e].min())] for c, s, e in peaks.itertuples(index=False, name=None)]
peak_counts = peaks['category'].value_counts().reindex(categories, fill_value=0)
genome_counts = sum((np.bincount(x, minlength=4) for x in genome_label.values()), np.zeros(4, dtype=np.int64))

summary = pd.DataFrame({'Peaks': peak_counts, 'Peak %': peak_counts / len(peaks) * 100,
                        'Genome bp': genome_counts, 'Genome %': genome_counts / genome_counts.sum() * 100})
display(summary.round(2))

colors = ['#2f5b8c', '#e31a1c', '#8c6bb1', '#16a85b']
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, values, title in zip(axes, [peak_counts, genome_counts], ['BAF60-binding sites', 'Genome fraction']):
    ax.pie(values, labels=categories, colors=colors, autopct='%1.1f%%', startangle=90)
    ax.set_title(title, fontweight='bold')
plt.tight_layout()